In [ ]:
import pandas as pd
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
import numpy as np
import pickle

DATA = "primekg"

df = pd.read_csv(f"/home/cc/cc_my_mounting_point/kg/{DATA}/kg.csv", sep=",", low_memory=False)
save_path = f"/home/cc/phd/KGEmbeddings/data/{DATA}/"

combinations = [
    ("drug_protein", "disease_protein", "pathway_protein"),        # 9.0
    ("drug_protein", "exposure_protein", "bioprocess_protein"),    # 8.0
    ("anatomy_protein_present", "disease_protein", "molfunc_protein"),  # 8.0
    ("disease_phenotype_positive", "phenotype_protein", "drug_effect"), # 7.5
    ("exposure_disease", "disease_disease", "indication"),         # 7.5
    ("drug_drug", "drug_protein", "contraindication"),             # 7.0
    ("disease_protein", "pathway_protein", "drug_effect"),         # 7.0
    ("phenotype_protein", "disease_protein", "pathway_protein"),   # 6.5
    ("indication", "off-label use", "drug_effect"),                # 6.0
    ("anatomy_protein_absent", "anatomy_protein_present", "cellcomp_protein"), # 6.0
    ("drug_effect", "drug_drug", "indication"),                    # 6.0
]

rel_set = {r for combo in combinations for r in combo}

df_filtered = df[df["relation"].isin(rel_set)]

In [ ]:

# x_counts = df_filtered["x_index"].value_counts()

# # --- Step 2: tag rows whose x_index appears < 4 times ---
# df_filtered["split_group"] = df_filtered["x_index"].map(lambda x: "rare" if x_counts[x] < 4 else "common")

# # Separate rare and common subsets
# df_rare = df_filtered[df_filtered["split_group"] == "rare"]
# df_common = df_filtered[df_filtered["split_group"] == "common"]

# # --- Step 3: split the common subset while maintaining relation proportions ---
# # Stratified splitting by relation ensures the relation distribution is preserved
# train_df, temp_df = train_test_split(
#     df_common,
#     test_size=0.30,   # 30% goes to val+test
#     stratify=df_common["relation"],
#     random_state=42
# )

# val_df, test_df = train_test_split(
#     temp_df,
#     test_size=0.50,   # split remaining 30% equally → 15% val, 15% test
#     stratify=temp_df["relation"],
#     random_state=42
# )

# # --- Step 4: add all rare x_index rows entirely to train ---
# train_df = pd.concat([train_df, df_rare], ignore_index=True)

# # --- STEP 4: Create node and relation mappings ---
# # Get unique node indices from x_index and y_index
# unique_nodes = pd.Index(sorted(set(df_filtered["x_index"]).union(set(df_filtered["y_index"]))))

# # Create mapping: node -> id (0..N-1)
# node2id = pd.Series(data=range(len(unique_nodes)), index=unique_nodes)

# # Create relation mapping: relation -> id
# unique_relations = pd.Index(sorted(df_filtered["relation"].unique()))
# rel2id = pd.Series(data=range(len(unique_relations)), index=unique_relations)

# # --- STEP 5: Map values to IDs in each split ---
# def map_to_ids(sub_df):
#     sub_df = sub_df.copy()
#     sub_df["head_id"] = sub_df["x_index"].map(node2id)
#     sub_df["relation_id"] = sub_df["relation"].map(rel2id)
#     sub_df["tail_id"] = sub_df["y_index"].map(node2id)
#     return sub_df[["head_id", "relation_id", "tail_id"]]

# train_mapped = map_to_ids(train_df)
# val_mapped = map_to_ids(val_df)
# test_mapped = map_to_ids(test_df)

# # --- STEP 6: Save the ID mappings (optional but recommended) ---
# node_map = pd.DataFrame({"node_index": unique_nodes, "node_id": range(len(unique_nodes))})
# rel_map = pd.DataFrame({"relation": unique_relations, "relation_id": range(len(unique_relations))})
# node_map.to_csv(f"{save_path}entity_map.csv", index=False)
# rel_map.to_csv(f"{save_path}relation_map.csv", index=False)

# # --- STEP 7: Save final triples for each split ---
# train_mapped.to_csv(f"{save_path}train.csv", index=False)
# val_mapped.to_csv(f"{save_path}valid.csv", index=False)
# test_mapped.to_csv(f"{save_path}test.csv", index=False)

# # --- STEP 8: Print summary ---
# print("✅ Data splits and mappings created successfully!\n")
# print(f"Nodes: {len(unique_nodes)}, Relations: {len(unique_relations)}")
# print(f"Train edges: {len(train_mapped)}, Val edges: {len(val_mapped)}, Test edges: {len(test_mapped)}")

| #  | R1                         | R2                         | Tail node type | R3 (tail → result) | Score | Reasoning                                                                                     |
| -- | -------------------------- | -------------------------- | -------------- | ------------------ | ----- | --------------------------------------------------------------------------------------------- |
| 1  | drug_protein               | disease_protein            | protein        | pathway_protein    | 9.0   | Very interpretable: drug target & disease-associated protein → which pathway it’s in.         |
| 2  | drug_protein               | exposure_protein           | protein        | bioprocess_protein | 8.0   | Drug & exposure both link to protein → what biological process is the protein in.             |
| 3  | anatomy_protein_present    | disease_protein            | protein        | molfunc_protein    | 8.0   | Anatomy + disease converge on a protein → then molecular function.                            |
| 4  | disease_phenotype_positive | phenotype_protein          | phenotype      | drug_effect        | 7.5   | Phenotype as tail: disease → phenotype, phenotype → protein, then drug effect on phenotype.   |
| 5  | exposure_disease           | disease_disease            | disease        | indication         | 7.5   | Exposure and disease/disease converge on a disease → then drugs indicated for that disease.   |
| 6  | drug_drug                  | drug_protein               | drug           | contraindication   | 7.0   | Drug interacts with another drug + has target protein → what contraindications occur.         |
| 7  | disease_protein            | pathway_protein            | protein        | drug_effect        | 7.0   | Disease & pathway converge on protein → then drug effects for that protein.                   |
| 8  | phenotype_protein          | disease_protein            | protein        | pathway_protein    | 6.5   | Two head relations linking to protein (phenotype→protein & disease→protein) → then pathway.   |
| 9  | indication                 | off-label use              | disease        | drug_effect        | 6.0   | Both head relations are drug→disease; tail = disease → then drug effects.                     |
| 10 | anatomy_protein_absent     | anatomy_protein_present    | protein        | cellcomp_protein   | 6.0   | Anatomy absence/presence → protein → then cellular component.                                 |
| 11 | drug_effect                | drug_drug                  | drug           | indication         | 6.0   | Drug effects + drug–drug → drug → what indications exist.                                     |
| 12 | molfunc_protein            | cellcomp_protein           | protein        | bioprocess_protein | 5.5   | Two protein-centred head relations (molecular function & cellular component) → process.       |
| 13 | exposure_cellcomp          | cellcomp_protein           | cellcomp       | molfunc_protein    | 5.5   | Exposure affects cell comp + cell comp→protein → then molecular function.                     |
| 14 | disease_disease            | disease_protein            | disease        | pathway_protein    | 5.5   | Disease & disease/protein converge → disease → then pathways via protein.                     |
| 15 | phenotype_phenotype        | disease_phenotype_negative | phenotype      | drug_effect        | 5.0   | Two phenotype relations converge → phenotype → drug effect.                                   |
| 16 | pathway_pathway            | pathway_protein            | pathway        | bioprocess_protein | 4.5   | Pathway & pathway/protein converge → pathway → then biological process.                       |
| 17 | anatomy_anatomy            | anatomy_protein_absent     | anatomy        | phenotype_protein  | 4.0   | Anatomy relations converge → anatomy → then phenotype via protein.                            |
| 18 | molfunc_molfunc            | cellcomp_cellcomp          | molfunc        | protein_protein    | 3.0   | Two “same-type” relations converge → molecular function → then protein–protein; likely dense. |


In [ ]:
# def normalize_type(t):
#     if t.lower() in {"gene/protein", "gene_protein"}:
#         return "protein"
#     elif t.lower() in {'effect/phenotype', 'effect_phenotype'}:
#         return "phenotype"
#     return t.lower()

# # Apply normalization
# df["x_type_norm"] = df["x_type"].apply(normalize_type)
# df["y_type_norm"] = df["y_type"].apply(normalize_type)

# # Define a function to check orientation matches relation
# def matches_orientation(row):
#     rel = row["relation"]
#     parts = rel.split("_")
#     if len(parts) < 2:
#         return False
#     sourceType = parts[0].lower()
#     targetType = parts[1].lower()
#     return (row["x_type_norm"] == sourceType and row["y_type_norm"] == targetType)

# df_oriented = df_filtered[df_filtered.apply(matches_orientation, axis=1)]

# print(f"Kept {len(df_oriented)} oriented edges out of {len(df_filtered)} total edges.")

In [ ]:
def compute_shared_tails(df_filtered, combinations, number_of_proj):
    # Pre-compute heads per tail per relation
    # Map: (relation, tail) → set of heads
    rel_tail_to_heads = (
        df_filtered
        .groupby(['relation', 'y_index'])['x_index']
        .agg(lambda s: set(s))
        .reset_index()
    )
    # Pivot or create a dict for quick lookup
    # relation → { tail → heads_set }
    rel_to_tail_heads = {}
    for rel in set(r for combo in combinations for r in combo):
        sub = rel_tail_to_heads[rel_tail_to_heads['relation'] == rel]
        rel_to_tail_heads[rel] = dict(zip(sub['y_index'], sub['x_index']))
    
    queries = []
    for (r1, r2, r_final) in tqdm(combinations , desc="Processing combinations"):
        print(f"Processing combination: {r1}, {r2} -> {r_final}")
        # Find tails that appear in both r1 and r2
        tails_r1 = set(rel_to_tail_heads.get(r1, {}).keys())
        tails_r2 = set(rel_to_tail_heads.get(r2, {}).keys())
        shared_tails = tails_r1.intersection(tails_r2)
        
        for tail in shared_tails:
            heads1 = list(rel_to_tail_heads[r1][tail])[:number_of_proj]
            heads2 = list(rel_to_tail_heads[r2][tail])[:number_of_proj]
            
            if not heads1 or not heads2:
                continue
            
            # Now find results for r_final: tail → results
            heads_for_final = rel_to_tail_heads.get(r_final, {}).get(tail, set())
            if not heads_for_final:
                continue
            
            # Build queries for cross-product of heads1 × heads2
            for h1 in heads1:
                for h2 in heads2:
                    queries.append({
                        "h1": h1,
                        "h2": h2,
                        "tail": tail,
                        "r1": r1,
                        "r2": r2,
                        "r_final": r_final,
                        "res": list(heads_for_final)
                    })

    np.random.shuffle(queries)
    return queries

In [ ]:
number_of_proj = 2

unique_nodes = pd.Index(sorted(set(df_filtered["x_index"]).union(set(df_filtered["y_index"]))))
node2id = pd.Series(data=range(len(unique_nodes)), index=unique_nodes)

df_filtered['x_index'] = df_filtered['x_index'].map(node2id)
df_filtered['y_index'] = df_filtered['y_index'].map(node2id)

queries = compute_shared_tails(df_filtered, combinations, number_of_proj)

# with open(f'/home/cc/phd/KGEmbeddings/queries/{DATA}/queries.pkl', 'wb') as f:
#     pickle.dump(queries, f)

len(queries)

In [ ]:
import pandas as pd
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
import numpy as np
import pickle

DATA = "primekg"
save_path = f"/home/cc/phd/KGEmbeddings/data/{DATA}/"

node_map = pd.read_csv(f"{save_path}entity_map.csv")
rel_map = pd.read_csv(f"{save_path}relation_map.csv")

with open(f'/home/cc/phd/KGEmbeddings/queries/{DATA}/queries.pkl', 'rb') as f:
    queries = pickle.load(f)

In [ ]:
query = queries[0]
query